# More Than A Hack 2027 – Offizielle Baseline reproduzieren

**Ziel:** Die vier veröffentlichten Baseline-Varianten aus dem lokalen Tactical-Tracking neu berechnen, mit den veröffentlichten Vorhersagen vergleichen und mit dem **unveränderten offiziellen Evaluator auf den öffentlichen fünf Minuten** bewerten.

Dieses Notebook ist unabhängig von Notebook 01 ausführbar. Es verändert keine Ballbesitzregeln, ergänzt keine Kontrollbestätigung, optimiert keine Schwellen und nutzt weder Video noch Smart Tags für die Vorhersage.

**Ausführung:** Python-Kernel mit `numpy`, `pandas`, `matplotlib` und `IPython` wählen → **Restart Kernel → Run All**. Es gibt keinen Netzwerkzugriff und keine automatische Paketinstallation beim Ausführen. Die Verarbeitung und alle eigenen analytischen Funktionen stehen im Notebook.

Die Originalreferenz liegt als unverändertes ZIP in `reference/`. Nur für den Aufruf des offiziellen Validators und Evaluators wird das Archiv vorübergehend in einen temporären Ordner entpackt und danach automatisch entfernt. Es entstehen keine eigenen externen Python-Module. Ergebnisse werden unter `output/baseline_reproduction/` gespeichert. Der Evaluator liest ausschließlich die erzeugten CSVs und die öffentliche Ground Truth.

**Datenfreigabe:** Das Archiv enthält das öffentliche Veranstalter-Repository, nicht die lizenzierten Matchdateien. Lokale Trackingdaten, Videobilder und daraus rekonstruierbare Daten gehören nicht in ein öffentliches Repository.

In [ ]:
# Standardbibliothek und Reproduzierbarkeit
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from tempfile import TemporaryDirectory
from importlib.metadata import version
import hashlib
import json
import subprocess
import sys
import xml.etree.ElementTree as ET

# Daten und Darstellung
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display, Markdown

## 1. Festgelegte Referenz und Konfiguration

Stand der Originalquellen: Commit [`3de6b60aeabd2475d95879a651d38b72bc4fba00`](https://github.com/metrica-sports/more-than-a-hack-2027/tree/3de6b60aeabd2475d95879a651d38b72bc4fba00). Das lokale Archiv wird per SHA-256 geprüft. Dadurch kann ein später geänderter GitHub-Stand die Reproduktion nicht unbemerkt verändern.

Die Spielfeldgröße **105,3 × 68 m** und alle vier Fenster sind exakt aus dieser offiziellen Referenz übernommen. Notebook 01 verwendete für reine Darstellungen die XML-Maße 105 × 68 m; hier ist die Originalbaseline maßgeblich. `MAIN_BASELINE` ist vor Betrachtung der Scores fest auf `smooth50` gesetzt, die vom Veranstalter benannte Referenz.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.xml").is_file())
ARCHIVE = PROJECT_DIR / "reference/mtah2027_official_3de6b60.zip"
COMMIT = "3de6b60aeabd2475d95879a651d38b72bc4fba00"
ARCHIVE_SHA256 = "019f497753d7f561d3fbb6048beef20b1bc1f244961db40d7c1914fa4ab76342"
META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
OUTPUT_DIR = PROJECT_DIR / "output/baseline_reproduction"
SUBMISSION_DIR, EVALUATION_DIR = OUTPUT_DIR / "submissions", OUTPUT_DIR / "evaluation"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

N_FRAMES, FPS, PITCH = 199319, 25, (105.3, 68.0)
VARIANTS = [("baseline_raw", 0), ("baseline_smooth25", 25),
            ("baseline_smooth50", 50), ("baseline_smooth125", 125)]
MAIN_BASELINE = "baseline_smooth50"
FIELD_SEP, GROUP_SEP, MISSING = ":", ";", "N"
HOME, AWAY, NOBODY = 0, 1, -1
LABEL_ORDER, COLORS = ["H", "A", "D"], ["#1764ab", "#c83e41", "#a0a0a0"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == ARCHIVE_SHA256, "Das Referenzarchiv wurde verändert."
print(f"Python {sys.version.split()[0]} | Referenz-Commit: {COMMIT}")
display(pd.Series({p: version(p) for p in ["numpy", "pandas", "matplotlib", "IPython"]}, name="Version").to_frame())
print("Keine Zufallsoperationen, kein Training und keine Parametersuche.")

## 2. Öffentliche Ground Truth und Teams prüfen

Die Referenzdatei enthält 199.319 Zeilen, aber nur im veröffentlichten Fenster echte Labels. **GT-`X` bedeutet hier: nicht bewerten.** Außerhalb dieses Fensters werden keine unbekannten Labels erfunden. Die Frame-IDs werden vor jedem Vergleich auf identische Reihenfolge geprüft.

Die ATD-XML nennt zuerst das Heimteam Barcelona und danach Athletic. Wie im Original werden die Track-Slots in ihrer XML-Reihenfolge einem Team zugeordnet. Spieleridentitäten und der Geschwindigkeitskanal werden nicht benötigt.

In [ ]:
with ZipFile(ARCHIVE) as official:
    ARCHIVE_ROOT = official.namelist()[0]
    gt = pd.read_csv(BytesIO(official.read(ARCHIVE_ROOT + "evaluation/gt/BAR-ATH_possession_gt_public.csv")))
    public_meta = json.loads(official.read(ARCHIVE_ROOT + "evaluation/gt/BAR-ATH_possession_gt_public_meta.json"))
    reference_predictions = {name: pd.read_csv(BytesIO(official.read(ARCHIVE_ROOT + f"evaluation/results/{name}.csv")))
                             for name, _ in VARIANTS}

assert np.array_equal(gt["frame"], np.arange(N_FRAMES)), "Ground Truth und Videoindex sind nicht ausgerichtet."
graded = gt["label"].ne("X").to_numpy()
public_frames = gt.loc[graded, "frame"].to_numpy()
root = ET.parse(META_PATH).getroot()
team_nodes = list(root.iter("Team"))
team_ids = [t.get("id") for t in team_nodes]
track_nodes = list(root.iter("Player"))
team_of_track = np.array([HOME if p.get("teamId") == team_ids[0] else AWAY for p in track_nodes], dtype=np.int8)

print(f"Öffentlich bewertet: {graded.sum():,} Frames = {graded.sum() / FPS:.0f} s; Video-Frames {public_frames[0]} … {public_frames[-1]}")
print(f"Metadaten: {public_meta['n_possessions']} Ballbesitzphasen; {public_meta['n_segments']} Segmente einschließlich D")
display(pd.DataFrame({"Label": ["H", "A"], "Team": [t.findtext("Name") for t in team_nodes],
                      "Track-Slots": [(team_of_track == side).sum() for side in [HOME, AWAY]]}))
display(gt.loc[graded, "label"].value_counts().reindex(LABEL_ORDER).rename("Öffentliche Frames").to_frame())

## 3. Originalalgorithmus: nächster Spieler zum Ball

Die folgende Funktion ist **wörtlich aus `starter/baseline.py` des festgelegten Commits übernommen**. Sie liest Rohpositionen als Python-Floats, skaliert beide Achsen in Meter und wählt den nächsten beobachteten Track-Slot. Roh-Frame 1 entspricht Video-Frame 0.

Fehlt der Ball oder lässt sich kein Spieler zuordnen, bleibt der interne Code `NOBODY`. Dieser wird bei der ungesmoothten Ausgabe zu `D`. Das ist eine bewusst einfache Annahme der Referenz und **keine bestätigte Erkennung von Spielunterbrechungen**. Auch das Überspringen fehlerhafter Zeilen und die Behandlung von Gleichständen bleiben wie im Original; dieses Notebook repariert die Methode nicht.

In [ ]:
def nearest_player_labels(raw_path, team_of_track, n_frames, pitch):
    """Per-frame team of the player closest to the ball. NOBODY where there is no answer.

    Returned as codes rather than letters because the smoothing below counts them.

    Coordinates are normalised 0-1 and the pitch is not square, so both axes are scaled to metres
    before distances are compared. Skipping that would stretch every distance along the short axis
    and quietly pick the wrong player near the touchlines.
    """
    length, width = pitch
    out = np.full(n_frames, NOBODY, dtype=np.int8)

    with open(raw_path) as handle:
        for line in handle:
            parts = line.split(FIELD_SEP)
            if len(parts) != 3:
                continue
            ball = parts[2]
            if ball[0] == MISSING:
                # No ball, no answer. 41% of graded frames land here -- see DATA.md. Leaving them
                # as NOBODY rather than guessing is what makes the D class mean anything at all.
                continue

            comma = ball.index(",")
            ball_x = float(ball[:comma]) * length
            ball_y = float(ball[comma + 1:ball.index(",", comma + 1)]) * width

            best, best_team = None, NOBODY
            for slot, group in enumerate(parts[1].split(GROUP_SEP)):
                if group[0] == MISSING:
                    continue
                comma = group.index(",")
                dx = float(group[:comma]) * length - ball_x
                dy = float(group[comma + 1:group.index(",", comma + 1)]) * width - ball_y
                distance = dx * dx + dy * dy
                if best is None or distance < best:
                    best, best_team = distance, team_of_track[slot]

            out[int(parts[0]) - 1] = best_team  # frame_id counts from 1, submissions from 0
    return out

In [ ]:
raw_codes = nearest_player_labels(RAW_PATH, team_of_track, N_FRAMES, PITCH)
raw_labels = np.where(raw_codes == HOME, "H", np.where(raw_codes == AWAY, "A", "D"))
print(f"Für {np.sum(raw_codes != NOBODY):,} von {N_FRAMES:,} Frames ist ein nächster Spieler zuordenbar.")
display(pd.DataFrame({"frame": public_frames[:8], "GT": gt.loc[graded, "label"].to_numpy()[:8],
                      "baseline_raw": raw_labels[graded][:8]}))

## 4. Die vier offiziellen Glättungsvarianten

Auch `smooth` ist wörtlich aus der Referenz übernommen. Der Parameter ist die **halbe Fensterbreite**: ±25 Frames = 51 Frames, ±50 = 101 Frames, ±125 = 251 Frames. Das Fenster ist zentriert und verwendet daher auch spätere Beobachtungen; diese Baseline arbeitet offline.

Wichtig: Die Abstimmung zählt nur H- und A-Beobachtungen. **`D` nimmt nicht an der Mehrheitswahl teil**; es wird nur ausgegeben, wenn im gesamten Fenster weder H noch A beobachtet wird. Ein Gleichstand geht an H. Es werden exakt die vier veröffentlichten Varianten berechnet, keine zusätzlichen Fenster ausprobiert.

In [ ]:
def smooth(codes, window):
    """Majority vote over a centred window of +/- `window` frames. D where nobody has the ball.

    Computed with cumulative sums, so the cost does not grow with the window -- a 125-frame window
    over 199319 frames is the same work as a 1-frame one.

    Ties go to the home side. With an odd-sized window over two classes a tie needs the two counts
    to be exactly equal, which is rare and never worth a coin flip.
    """
    if window <= 0:
        return np.where(codes == HOME, "H", np.where(codes == AWAY, "A", "D"))

    home_cumulative = np.concatenate(([0], np.cumsum(codes == HOME)))
    away_cumulative = np.concatenate(([0], np.cumsum(codes == AWAY)))

    index = np.arange(len(codes))
    low = np.maximum(0, index - window)
    high = np.minimum(len(codes), index + window + 1)
    home = home_cumulative[high] - home_cumulative[low]
    away = away_cumulative[high] - away_cumulative[low]

    labels = np.full(len(codes), "D", dtype="<U1")
    seen = (home + away) > 0
    labels[seen] = np.where(home[seen] >= away[seen], "H", "A")
    return labels

In [ ]:
predictions = {name: smooth(raw_codes, window) for name, window in VARIANTS}
configuration = pd.DataFrame([
    {"Variante": name, "Halbfenster (Frames)": window, "Fenster (Frames)": 2 * window + 1,
     "Fenster (s)": (2 * window + 1) / FPS} for name, window in VARIANTS
])
display(configuration)

## 5. Reproduktion gegen veröffentlichte Vorhersagen prüfen und exportieren

Für jede Variante vergleichen wir **alle 199.319 neu berechneten Labels** mit der entsprechenden veröffentlichten CSV. Das ist ein Test der Reproduktion, keine Evaluation gegen die private Ground Truth: Die vier Dateien enthalten lediglich die veröffentlichten Baseline-Vorhersagen.

Danach schreiben wir das offizielle Format `frame,label`, ab Frame 0 und mit genau einer Zeile pro Videoframe. Zusätzlich vergleichen wir die CSV-Dateien byteweise. Eine Labelabweichung stoppt den Durchlauf; wir passen keine Vorhersage an die Referenz an.

In [ ]:
reproduction_rows = []
with ZipFile(ARCHIVE) as official:
    for name, _ in VARIANTS:
        reference = reference_predictions[name]
        assert np.array_equal(reference["frame"], gt["frame"]), f"Abweichende Frame-Reihenfolge: {name}"
        differences = int(np.sum(predictions[name] != reference["label"].to_numpy()))
        assert differences == 0, f"{name}: {differences} Labels weichen von der Originalbaseline ab."
        submission = pd.DataFrame({"frame": np.arange(N_FRAMES), "label": predictions[name]})
        path = SUBMISSION_DIR / f"{name}.csv"
        submission.to_csv(path, index=False, lineterminator="\n")
        same_bytes = path.read_bytes() == official.read(ARCHIVE_ROOT + f"evaluation/results/{name}.csv")
        reproduction_rows.append({"Variante": name, "Abweichende Labels": differences,
                                  "CSV bytegleich": same_bytes, "Zeilen": len(submission),
                                  "SHA256": hashlib.sha256(path.read_bytes()).hexdigest()})

reproduction = pd.DataFrame(reproduction_rows)
display(reproduction.drop(columns="SHA256"))
reproduction.to_csv(OUTPUT_DIR / "reproduction_check.csv", index=False)

## 6. Offiziellen Validator und Evaluator ausführen

Wir verwenden die Originalprogramme direkt aus dem geprüften Archiv, einschließlich ihrer Metrikdefinitionen. Der temporäre Ordner dient nur der Ausführung dieser Referenzabhängigkeit. Eigene Analysefunktionen bleiben im Notebook. Der subprocess-Aufruf benutzt denselben Python-Interpreter wie der Notebook-Kernel.

Die Evaluation verwendet ausschließlich die **öffentliche** GT. Die Standardeinstellungen einschließlich `bootstrap=0` bleiben unverändert. Es gibt keine Parametersuche, keine Modellwahl und keinen Zugriff auf private Labels. Der offizielle Evaluator prüft das Format nochmals und schreibt `scoreboard.csv` sowie Berichte je Variante.

In [ ]:
validation_rows = []
with TemporaryDirectory(prefix="mtah2027-official-") as temporary:
    with ZipFile(ARCHIVE) as official:
        official.extractall(temporary)
    official_root = Path(temporary) / ARCHIVE_ROOT
    for name, _ in VARIANTS:
        validation = subprocess.run(
            [sys.executable, str(official_root / "evaluation/validate_submission.py"), str(SUBMISSION_DIR / f"{name}.csv")],
            capture_output=True, text=True, check=True)
        validation_rows.append({"Variante": name, "Offizieller Validator": validation.stdout.splitlines()[0]})

    evaluation = subprocess.run(
        [sys.executable, str(official_root / "evaluation/run_evaluation.py"), "--gt", "public",
         "--results", str(SUBMISSION_DIR), "--out", str(EVALUATION_DIR), "--no-detail"],
        capture_output=True, text=True, check=True)

(EVALUATION_DIR / "official_stdout.txt").write_text(evaluation.stdout, encoding="utf-8")
display(pd.DataFrame(validation_rows))
print(evaluation.stdout)

## 7. Scores lesen und mit den publizierten öffentlichen Werten vergleichen

`spellF1` ist der Mittelwert des Phasen-F1 über zeitliche IoU-Schwellen 0,3 bis 0,7. `macroF1` mittelt die frameweisen F1-Werte für H/A/D und ist der erste Tie-Breaker. D-Abschnitte sind selbst keine Ballbesitzphasen, trennen aber H-/A-Phasen.

Die folgenden Vergleichswerte sind die auf vier Dezimalstellen veröffentlichten **öffentlichen** Ergebnisse aus dem README desselben Referenzstandes. Die privaten Vollspiel-Scores werden hier nicht als selbst gemessene Ergebnisse ausgegeben.

In [ ]:
scoreboard = pd.read_csv(EVALUATION_DIR / "scoreboard.csv")
results = scoreboard.set_index("submission").loc[[name for name, _ in VARIANTS]]
published = pd.DataFrame.from_dict({
    "baseline_raw": {"published_macroF1": 0.5972, "published_spellF1": 0.0378},
    "baseline_smooth25": {"published_macroF1": 0.6200, "published_spellF1": 0.2868},
    "baseline_smooth50": {"published_macroF1": 0.6207, "published_spellF1": 0.3256},
    "baseline_smooth125": {"published_macroF1": 0.5812, "published_spellF1": 0.4242},
}, orient="index")
comparison = results[["macro_f1", "spell_f1_mean", "poss", "dur"]].join(published)
comparison["Publizierte Scores reproduziert"] = (
    comparison["macro_f1"].round(4).eq(comparison["published_macroF1"])
    & comparison["spell_f1_mean"].round(4).eq(comparison["published_spellF1"]))
display(comparison.round(4))
assert comparison["Publizierte Scores reproduziert"].all(), "Die gemessenen öffentlichen Scores weichen von der Dokumentation ab."

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
results[["macro_f1", "spell_f1_mean"]].rename(columns={"macro_f1": "macroF1", "spell_f1_mean": "spellF1"}).plot.bar(
    ax=axes[0], color=["#8aaac9", "#1764ab"], rot=20)
axes[0].set(title="Öffentliche fünf Minuten: zwei Sichtweisen auf Qualität", ylabel="Score", xlabel="", ylim=(0, 1))
axes[1].bar(results.index, results["poss"], color="#1764ab")
axes[1].axhline(public_meta["n_possessions"], color="#c83e41", linestyle="--", label="Ground Truth: 17 Phasen")
axes[1].set(title="Wie viele Ballbesitzphasen werden erkannt?", ylabel="Anzahl H-/A-Phasen", xlabel="")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()
plt.show()
display(Markdown(
    f"**Beobachtung:** Die Rohbaseline erzeugt auf dem öffentlichen Ausschnitt **{int(results.loc['baseline_raw', 'poss'])}** Phasen, "
    f"gegenüber **{public_meta['n_possessions']}** echten. `smooth50` erzeugt **{int(results.loc[MAIN_BASELINE, 'poss'])}**. "
    "Glättung reduziert die Fragmentierung; eine ähnliche Phasenanzahl allein bestätigt aber noch keine richtigen Phasengrenzen."))

## 8. Vorhersagen als Zeitverlauf ansehen

Die Zeilen zeigen dieselben fünf Minuten. Die Zeitachse beginnt beim ersten öffentlich gelabelten Frame. So werden kurze Fehlwechsel, falsche Unterbrechungen und verschobene Grenzen direkt sichtbar. Die Auswahl der Varianten wurde vor der Bewertung festgelegt.

In [ ]:
row_names = ["Ground Truth"] + [name for name, _ in VARIANTS]
label_rows = [gt.loc[graded, "label"].to_numpy()] + [predictions[name][graded] for name, _ in VARIANTS]
label_to_number = {label: number for number, label in enumerate(LABEL_ORDER)}
timeline = np.array([[label_to_number[label] for label in row] for row in label_rows])
fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
ax.imshow(timeline, interpolation="nearest", aspect="auto", cmap=ListedColormap(COLORS), vmin=0, vmax=2,
          extent=(0, graded.sum() / FPS, len(row_names) - 0.5, -0.5))
ax.set_yticks(np.arange(len(row_names)), row_names)
ax.set(title="Ground Truth und offizielle Baseline-Varianten im direkten Zeitvergleich", xlabel="Sekunden seit Beginn des öffentlichen Fensters")
ax.legend(handles=[Patch(color=color, label=label) for color, label in zip(COLORS, LABEL_ORDER)],
          loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.show()

**Interpretation:** Kleine andersfarbige Abschnitte zerteilen eine Phase. Breite D-Abschnitte zeigen hingegen Unterbrechungen an. Durch Glättung können solche Abschnitte sowohl verschwinden als auch zeitlich verschoben werden. Ob ein konkreter Wechsel richtig ist, zeigt hier die GT-Zeile; aus der geglätteten Darstellung allein lässt sich keine Ballkontrolle ableiten.

## 9. Fehler der vorab festgelegten Referenz `smooth50` untersuchen

Wir betrachten H/A/D-Verwechslungen und die längsten zusammenhängenden Abschnitte mit konstantem GT-/Vorhersagepaar. Die Tabelle dient als Einstieg in eine spätere Szenensichtung; sie benennt keine vermeintlichen Annahmen, Zweikämpfe oder Wiederanpfiffe ohne Videoprüfung. Außerhalb des öffentlichen Fensters lassen sich Fehler nicht gegen diese GT bestimmen.

In [ ]:
y_true = gt.loc[graded, "label"].to_numpy()
y_pred = predictions[MAIN_BASELINE][graded]
confusion = pd.crosstab(pd.Categorical(y_true, categories=LABEL_ORDER),
                        pd.Categorical(y_pred, categories=LABEL_ORDER), dropna=False)
confusion.index.name, confusion.columns.name = "Ground Truth", "Vorhersage"
row_percent = confusion.div(confusion.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
image = ax.imshow(row_percent, cmap="Blues", vmin=0, vmax=100)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{confusion.iloc[i, j]}\n{row_percent.iloc[i, j]:.1f} %", ha="center", va="center",
                color="white" if row_percent.iloc[i, j] > 55 else "black")
ax.set(xticks=range(3), xticklabels=LABEL_ORDER, yticks=range(3), yticklabels=LABEL_ORDER,
       xlabel="Vorhersage", ylabel="Ground Truth", title=f"{MAIN_BASELINE}: Fehler nach Klasse")
fig.colorbar(image, ax=ax, label="Anteil innerhalb der GT-Klasse (%)")
plt.show()
display(results.loc[[MAIN_BASELINE], ["f1_H", "f1_A", "f1_D"]].round(4))

In [ ]:
public_diagnostics = pd.DataFrame({"frame": public_frames, "GT": y_true, "prediction": y_pred})
pair_changes = (public_diagnostics[["GT", "prediction"]] != public_diagnostics[["GT", "prediction"]].shift()).any(axis=1).cumsum()
error_sections = public_diagnostics.groupby(pair_changes).agg(
    start_frame=("frame", "first"), end_frame=("frame", "last"), GT=("GT", "first"),
    prediction=("prediction", "first"), n_frames=("frame", "size"))
error_sections = error_sections.loc[error_sections["GT"].ne(error_sections["prediction"])].copy()
error_sections["duration_s"] = error_sections["n_frames"] / FPS
error_sections["start_video_s"] = error_sections["start_frame"] / FPS
error_sections["start_public_s"] = (error_sections["start_frame"] - public_frames[0]) / FPS
display(error_sections.nlargest(6, "duration_s").drop(columns="n_frames").round(2))

missed_dead_s = ((y_true == "D") & (y_pred != "D")).sum() / FPS
false_dead_s = ((y_true != "D") & (y_pred == "D")).sum() / FPS
display(Markdown(f"**Beobachtung:** `smooth50` sagt auf **{missed_dead_s:.2f} s** echter Unterbrechungen H oder A voraus. Umgekehrt sagt es auf **{false_dead_s:.2f} s** lebenden Spiels D voraus. Die Rohannahme ‚keine zuordenbare Beobachtung = D‘ und deren Glättung sind daher keine verlässliche Unterbrechungserkennung."))

## 10. Ergebnis und Grenzen

Die Reproduktion wird getrennt von der Güte des Verfahrens bewertet: Übereinstimmung mit der Referenz zeigt, dass wir dieselbe Methode ausgeführt haben. Sie zeigt nicht, dass diese Methode guten oder echten kontrollierten Ballbesitz erkennt.

In [ ]:
display(Markdown(
    f"- **Reproduktion:** Alle vier Varianten stimmen in sämtlichen **{N_FRAMES:,} Labels** mit der veröffentlichten Referenz überein.\n"
    f"- **Format:** Alle vier CSVs haben den offiziellen Validator bestanden.\n"
    f"- **Öffentliche Evaluation:** Die publizierten Scores aller vier Varianten wurden auf vier Dezimalstellen reproduziert.\n"
    f"- **Vorab gewählte Referenz `{MAIN_BASELINE}`:** macroF1 **{results.loc[MAIN_BASELINE, 'macro_f1']:.4f}**, "
    f"spellF1 **{results.loc[MAIN_BASELINE, 'spell_f1_mean']:.4f}**.\n"
    "- **Keine Erweiterung:** Ballkontrolle, Smart Tags, Videoauswertung und neue Regeln wurden nicht ergänzt."
))
manifest = {"official_commit": COMMIT, "archive_sha256": ARCHIVE_SHA256,
            "python": sys.version.split()[0], "packages": {p: version(p) for p in ["numpy", "pandas", "matplotlib", "IPython"]},
            "n_frames": N_FRAMES, "fps": FPS, "pitch_m": list(PITCH), "variants": dict(VARIANTS),
            "ground_truth": "public", "main_baseline": MAIN_BASELINE, "bootstrap": 0}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Vorhersagen: {SUBMISSION_DIR}\nBewertung und Originalberichte: {EVALUATION_DIR}")

**Einordnung:** Auf den öffentlichen fünf Minuten erreicht `smooth125` den höchsten spellF1. Das ist kein ausreichender Grund, diese Variante als besseres Verfahren auszuwählen: Die Referenzdokumentation zeigt bereits eine andere Reihenfolge auf dem privaten Vollspiel. Wir bleiben für die weitere Diagnose bei der vorab festgelegten `smooth50`-Referenz. 7.500 benachbarte Frames sind hier nur 17 Ballbesitzphasen; es werden weder statistische Überlegenheit noch Vollspielqualität aus diesem Ausschnitt behauptet.

**Nächster Schritt zur gemeinsamen Entscheidung:** Zunächst einige der gezeigten Fehlerabschnitte im taktischen Video ansehen. Danach könnten wir entweder die Unterbrechungserkennung verbessern oder eine Regel für bestätigte Ballkontrolle formulieren. Beides wären neue methodische Schritte; sie sind hier noch nicht umgesetzt.

Quellen des verwendeten Commits: [Baseline](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/starter/baseline.py), [öffentliche Referenzscores](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/README.md), [Evaluator](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/evaluation/run_evaluation.py), [Metrikdefinitionen](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/evaluation/possession_metrics.py).

## Lizenzhinweis zur übernommenen Originalbaseline und Referenz

Die Funktionen `nearest_player_labels` und `smooth` stammen von Metrica Sports. Das Originalarchiv enthält die vollständigen Quellen und den folgenden Lizenztext:

```text
SCOPE OF THIS LICENCE
---------------------

The MIT Licence below covers the contents of this repository only: the source code, the
documentation, and the public 5-minute ground truth in evaluation/gt/.

It does NOT cover the match data — the videos, the tracking data, the Smart Tagging and the
homography. Those assets are separately licensed for this competition only, under the terms you
accept at enrolment, and nothing in the licence below grants any right to use or redistribute
them. See DATA.md and RULES.md.

---

MIT License

Copyright (c) 2026-2027 Metrica Sports

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

```